## 1. Configurações e carregamento do dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

dataset_file = '../data/500+.csv'
ano_predicao = 2025

def filtrar_inconsistencias(df_data):
    return df_data.loc[(df_data['Artista'] != '???') & (df_data['Musica'].str.len() > 0) & (df_data['Observacao'] != 'repetida')]

def load_data(agregar_pinkfloyd):
    df_data = pd.read_csv(dataset_file)
    df_data['Data_Lancamento_Album'] = pd.to_datetime(df_data['Data_Lancamento_Album'])
    df_data['Decada_Musica'] = (df_data['Data_Lancamento_Album'].dt.year // 10) * 10
    df_data['Ano_Musica'] = df_data['Data_Lancamento_Album'].dt.year
    df_data['Duracao'] = df_data.loc[:,'Duracao'].fillna(value=0)
    if (agregar_pinkfloyd):
        df_data.loc[df_data['Musica'].str.contains('Another Brick', na=False), 'Musica'] = 'Another Brick in the Wall'
        df_data.loc[df_data['Musica'].str.contains('Another Brick', na=False), 'Duracao'] = 508

    df_data = df_data.drop(['Artista_Origem', 'Musica_Origem', 'Artista_Wikidata_ID', 'Artista_Wikidata', 'Artista_Wiki', 'Country', 'Genre', 'Musica_Wikidata_ID', 'Musica_Wikidata', 'Musica_Wiki', 'Album_Single_Wikidata_ID', 'Album_Single_Wikidata', 'Album_Single_Wiki', 'Data_Lancamento_Album'], axis=1)
    df_data.rename(columns={'Album_Single':'Album'}, inplace=True)
    df_data = filtrar_inconsistencias(df_data)
    return df_data


df = load_data(True)
df = df[df['Ano'] < ano_predicao]
print("Dataset carregado com sucesso!")

## 2. Identificador única da música

In [ ]:
# Criar identificador único baseado na tríplice Artista-Musica-Observacao
df['Observacao_filled'] = df['Observacao'].fillna('NONE')
df['ID_Musica'] = df['Artista'] + '|||' + df['Musica'] + '|||' + df['Observacao_filled']

print(f"\nTotal de músicas únicas: {df['ID_Musica'].nunique()}")
print(f"Total de registros: {len(df)}")

## 3. Engenharia de features temporais

In [ ]:
df_sorted = df.sort_values(['ID_Musica', 'Ano'])
features_list = []

print("Criando features históricas...")

for musica_id in df['ID_Musica'].unique():
    df_musica = df_sorted[df_sorted['ID_Musica'] == musica_id].copy()
    
    for idx, row in df_musica.iterrows():
        ano_atual = row['Ano']
        historico = df_musica[df_musica['Ano'] < ano_atual]
        
        features = {
            'ID_Musica': musica_id,
            'Ano': ano_atual,
            'Posicao': row['Posicao'],
            'Artista': row['Artista'],
            'Musica': row['Musica'],
            'Pais': row['Pais'],
            'Genero': row['Genero'],
            'Duracao': row['Duracao'],
            'Ano_Musica': row['Ano_Musica'],
            'Decada_Musica': row['Decada_Musica'],
            
            # Features históricas
            'num_aparicoes': len(historico),
            'melhor_posicao': historico['Posicao'].min() if len(historico) > 0 else np.nan,
            'pior_posicao': historico['Posicao'].max() if len(historico) > 0 else np.nan,
            'posicao_media': historico['Posicao'].mean() if len(historico) > 0 else np.nan,
            'posicao_std': historico['Posicao'].std() if len(historico) > 0 else np.nan,
            'anos_desde_primeira': ano_atual - historico['Ano'].min() if len(historico) > 0 else 0,
            'apareceu_ano_anterior': 1 if (ano_atual - 1) in historico['Ano'].values else 0,
            'posicao_ano_anterior': historico[historico['Ano'] == ano_atual - 1]['Posicao'].values[0] 
                                    if (ano_atual - 1) in historico['Ano'].values else np.nan,
            'tendencia_posicao': None,  # Calcular depois
            'idade_musica': ano_atual - row['Ano_Musica']
        }
        
        # Calcular tendência (melhoria ou piora nas últimas aparições)
        if len(historico) >= 2:
            ultimas_posicoes = historico.tail(3)['Posicao'].values
            if len(ultimas_posicoes) >= 2:
                features['tendencia_posicao'] = ultimas_posicoes[-1] - ultimas_posicoes[0]
        
        features_list.append(features)
    
df_features = pd.DataFrame(features_list)
print(f"Features criadas com sucesso! Shape: {df_features.shape}")
print(f"\nColunas disponíveis:")
print(df_features.columns.tolist())

## 4. Engenharia de features de aparição

In [ ]:
print("Criando features de aparição...")

ano_limite = ano_predicao - 1
df_historico = df_features[df_features['Ano'] <= ano_limite].copy()

# Obter todas as músicas únicas que já apareceram
musicas_unicas = df_historico['ID_Musica'].unique()

features_atualizadas = []

for musica_id in musicas_unicas:
    df_musica = df_historico[df_historico['ID_Musica'] == musica_id].sort_values('Ano')
    
    # Pegar informações da última aparição
    ultima_aparicao = df_musica.iloc[-1]
    
    # Calcular features baseadas em TODO o histórico
    features = {
        'ID_Musica': musica_id,
        'Artista': ultima_aparicao['Artista'],
        'Musica': ultima_aparicao['Musica'],
        'Musica': ultima_aparicao['Musica'],
        'Pais': ultima_aparicao['Pais'],
        'Genero': ultima_aparicao['Genero'],
        'Duracao': ultima_aparicao['Duracao'],
        'Ano_Musica': ultima_aparicao['Ano_Musica'],
        'Decada_Musica': ultima_aparicao['Decada_Musica'],
        
        # FEATURES HISTÓRICAS (considerando TUDO até ano_limite)
        'num_aparicoes': len(df_musica),
        'melhor_posicao': df_musica['Posicao'].min(),
        'pior_posicao': df_musica['Posicao'].max(),
        'posicao_media': df_musica['Posicao'].mean(),
        'posicao_std': df_musica['Posicao'].std() if len(df_musica) > 1 else 0,
        'anos_desde_primeira': ano_limite - df_musica['Ano'].min(),
        'apareceu_ano_anterior': 1 if ano_limite in df_musica['Ano'].values else 0,
        'posicao_ano_anterior': df_musica[df_musica['Ano'] == ano_limite]['Posicao'].values[0] 
                                if ano_limite in df_musica['Ano'].values else np.nan,
        'Posicao': ultima_aparicao['Posicao'],  # Última posição conhecida
        'idade_musica': ano_limite - ultima_aparicao['Ano_Musica'],
        'anos_desde_ultima': ano_limite - df_musica['Ano'].max(),
        
        # Tendência das últimas 3 aparições
        'tendencia_posicao': 0
    }
    
    # Calcular tendência
    if len(df_musica) >= 2:
        ultimas_posicoes = df_musica.tail(3)['Posicao'].values
        if len(ultimas_posicoes) >= 2:
            features['tendencia_posicao'] = ultimas_posicoes[-1] - ultimas_posicoes[0]
    
    # Calcular frequência de aparição (% anos que apareceu)
    anos_possiveis = ano_limite - df_musica['Ano'].min() + 1
    features['frequencia_aparicao'] = len(df_musica) / anos_possiveis if anos_possiveis > 0 else 0
    
    # Verificar se está em "streak" (apareceu nos últimos N anos consecutivos)
    streak = 0
    for ano_check in range(ano_limite, ano_limite - 5, -1):
        if ano_check in df_musica['Ano'].values:
            streak += 1
        else:
            break
    features['streak_anos'] = streak
    
    # ===== NOVAS FEATURES PARA TRATAR OUTLIERS =====
    
    # Flag: música apareceu apenas 1 vez?
    features['aparicao_unica'] = 1 if len(df_musica) == 1 else 0
    
    # Se apareceu apenas 1 vez, há quantos anos foi?
    features['anos_desde_unica_aparicao'] = ano_limite - df_musica['Ano'].max() if len(df_musica) == 1 else 0
    
    # Taxa de "dropout" - chance de nunca mais voltar após primeira aparição
    # Músicas que aparecem 1x e nunca voltam têm alta taxa de dropout
    if len(df_musica) == 1:
        anos_decorridos = ano_limite - df_musica['Ano'].max()
        # Se passou 1 ano: dropout_score = 1, 2 anos = 2, etc
        features['dropout_score'] = anos_decorridos
    else:
        features['dropout_score'] = 0
    
    # Consistência: razão entre aparições e anos possíveis
    # Valores baixos indicam aparições esporádicas
    features['consistencia'] = features['frequencia_aparicao']
    
    # Volatilidade de posição (amplitude)
    if len(df_musica) > 1:
        features['volatilidade_posicao'] = features['pior_posicao'] - features['melhor_posicao']
    else:
        features['volatilidade_posicao'] = 0
    
    # Força da música: combinação de múltiplos fatores
    # Quanto maior, mais "estabelecida" é a música
    forca = 0
    if len(df_musica) >= 3:
        forca += 3  # Apareceu múltiplas vezes
    if features['frequencia_aparicao'] > 0.5:
        forca += 2  # Alta frequência
    if features['streak_anos'] >= 2:
        forca += 2  # Em streak
    if features['melhor_posicao'] <= 100:
        forca += 1  # Já esteve no top 100
    
    features['forca_musica'] = forca
    
    # Penalidade para "one-hit wonders" (músicas que aparecem 1x e param)
    # Quanto maior o tempo desde a única aparição, maior a penalidade
    if features['aparicao_unica'] == 1:
        features['penalidade_one_hit'] = min(5, features['anos_desde_unica_aparicao'])
    else:
        features['penalidade_one_hit'] = 0
    
    features_atualizadas.append(features)

features_atualizadas = pd.DataFrame(features_atualizadas)

print(f"Features criadas com sucesso! Shape: {features_atualizadas.shape}")
print(f"\nColunas disponíveis:")
print(features_atualizadas.columns.tolist())

## 5. Probabilidade de aparição a cada ano

In [ ]:
print("Calculando probabilidades de aparição...")

# Preparar dados para cada música em cada ano
resultados = []

for musica_id in df_features['ID_Musica'].unique():
    df_musica = df_features[df_features['ID_Musica'] == musica_id].sort_values('Ano')
    
    for i in range(len(df_musica) - 1):
        ano_atual = df_musica.iloc[i]['Ano']
        ano_seguinte = ano_atual + 1
        apareceu_seguinte = 1 if ano_seguinte in df_musica['Ano'].values else 0
        
        resultado = df_musica.iloc[i].to_dict()
        resultado['apareceu_proximo_ano'] = apareceu_seguinte
        resultados.append(resultado)


df_prob = pd.DataFrame(resultados)

# Estatísticas gerais
taxa_aparicao_geral = df_prob['apareceu_proximo_ano'].mean()
print(f"\nTaxa geral de aparição no próximo ano: {taxa_aparicao_geral:.2%}")

# Análise por número de aparições
print("\nTaxa de aparição por histórico:")
for n in range(1, 6):
    mask = df_prob['num_aparicoes'] == n
    if mask.sum() > 0:
        taxa = df_prob[mask]['apareceu_proximo_ano'].mean()
        print(f"  {n} aparições anteriores: {taxa:.2%} (n={mask.sum()})")

## 6. Modelo para cálculo de probabilidade de aparição

In [ ]:
# Selecionar features numéricas
feature_cols_aparicao = ['num_aparicoes', 'melhor_posicao', 'pior_posicao', 
                'posicao_media', 'anos_desde_primeira', 'apareceu_ano_anterior',
                'Duracao', 'idade_musica', 'Posicao']

# Preparar dados
df_train = df_prob[df_prob['Ano'] < ano_predicao - 2].copy()
df_test = df_prob[df_prob['Ano'] >= ano_predicao - 2].copy()

# Remover NaNs
df_train = df_train.dropna(subset=feature_cols_aparicao + ['apareceu_proximo_ano'])
df_test = df_test.dropna(subset=feature_cols_aparicao)

X_train = df_train[feature_cols_aparicao]
y_train = df_train['apareceu_proximo_ano']
X_test = df_test[feature_cols_aparicao]

# Treinar modelo
modelo_aparicao = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
modelo_aparicao.fit(X_train, y_train)

# Importância das features
feature_importance = pd.DataFrame({
    'feature': feature_cols_aparicao,
    'importance': modelo_aparicao.feature_importances_
}).sort_values('importance', ascending=False)

print("\nImportância das features para aparição:")
print(feature_importance.to_string(index=False))

## 7. Geração de probabilidades de aparição

In [ ]:
print(f"{'='*80}")
print(f"CALCULANDO PREDIÇÕES PARA {ano_predicao}")
print(f"Considerando TODO o histórico até {ano_predicao - 1}")
print(f"{'='*80}")

# Recalcular features baseado em todo histórico até o ano anterior
df_atual = features_atualizadas

print(f"\nTotal de músicas no histórico: {len(df_atual)}")
print(f"Músicas que apareceram em {ano_predicao - 1}: {df_atual['apareceu_ano_anterior'].sum()}")
print(f"Músicas com apenas 1 aparição: {df_atual['aparicao_unica'].sum()}")

predicoes = []

for _, row in df_atual.iterrows():
    try:
        # Preparar features para aparição (incluindo as novas)
        features_disponiveis = [f for f in feature_cols_aparicao if f in row.index]
        X_aparicao = row[features_disponiveis].values.reshape(1, -1)
        X_aparicao = np.nan_to_num(X_aparicao, nan=0)
        
        # Prever probabilidade de aparição
        prob_aparicao_raw = modelo_aparicao.predict_proba(X_aparicao)[0][1]
        
        # ===== APLICAR AJUSTES PARA OUTLIERS =====
        prob_aparicao = prob_aparicao_raw
        
        # REGRA 1: Penalizar músicas com apenas 1 aparição
        if row['aparicao_unica'] == 1:
            anos_desde = row['anos_desde_unica_aparicao']
            
            # Penalidade crescente: 1 ano = 20%, 2 anos = 40%, 3+ anos = 60%
            if anos_desde == 0:  # Apareceu no ano anterior
                penalidade = 0.20
            elif anos_desde == 1:
                penalidade = 0.40
            elif anos_desde == 2:
                penalidade = 0.55
            else:
                penalidade = 0.70
            
            prob_aparicao *= (1 - penalidade)
        
        # REGRA 2: Bonus para músicas consistentes (múltiplas aparições)
        if row['num_aparicoes'] >= 3 and row['frequencia_aparicao'] > 0.5:
            bonus = min(0.15, row['frequencia_aparicao'] * 0.2)
            prob_aparicao = min(0.99, prob_aparicao * (1 + bonus))
        
        # REGRA 3: Penalizar músicas que não aparecem há muito tempo
        if row['anos_desde_ultima'] >= 3:
            penalidade_ausencia = min(0.50, row['anos_desde_ultima'] * 0.10)
            prob_aparicao *= (1 - penalidade_ausencia)
        
        # REGRA 4: Bonus para streak (apareceu nos últimos anos consecutivos)
        if row['streak_anos'] >= 2:
            bonus_streak = min(0.20, row['streak_anos'] * 0.05)
            prob_aparicao = min(0.99, prob_aparicao * (1 + bonus_streak))
        
        # REGRA 5: Ajuste baseado na força geral da música
        if row['forca_musica'] >= 5:
            prob_aparicao = min(0.99, prob_aparicao * 1.10)
        elif row['forca_musica'] == 0 and row['aparicao_unica'] == 1:
            prob_aparicao *= 0.5  # Penalidade severa para one-hits fracos
        
        predicao = {
            'ID_Musica': row['ID_Musica'],
            'Artista': row['Artista'],
            'Musica': row['Musica'],
            'Genero': row['Genero'],
            'Pais': row['Pais'],
            'prob_aparicao_raw': prob_aparicao_raw,  # Probabilidade antes dos ajustes
            'prob_aparicao': prob_aparicao,  # Probabilidade ajustada
            'num_aparicoes_historicas': row['num_aparicoes'],
            'apareceu_ano_anterior': row['apareceu_ano_anterior'],
            'frequencia_aparicao': row['frequencia_aparicao'],
            'streak_anos': row['streak_anos'],
            'anos_desde_primeira': row['anos_desde_primeira'],
            'aparicao_unica': row['aparicao_unica'],
            'forca_musica': row['forca_musica'],
            'anos_desde_ultima': row['anos_desde_ultima']
        }
        
        predicoes.append(predicao)
        
    except Exception as e:
        print(f"Erro ao processar música {row['ID_Musica']}: {e}")
        continue

df_predicoes = pd.DataFrame(predicoes)
df_predicoes = df_predicoes.sort_values('prob_aparicao', ascending=False)


print(f"\nTotal de músicas analisadas: {len(df_predicoes)}")
print(f"Músicas com probabilidade > 50%: {(df_predicoes['prob_aparicao'] > 0.5).sum()}")
print(f"Músicas com probabilidade > 70%: {(df_predicoes['prob_aparicao'] > 0.7).sum()}")
print(f"Músicas com probabilidade > 90%: {(df_predicoes['prob_aparicao'] > 0.9).sum()}")

# Análise de one-hit wonders
one_hits = df_predicoes[df_predicoes['aparicao_unica'] == 1]
print(f"\n{'='*80}")
print(f"ANÁLISE DE ONE-HIT WONDERS (músicas com apenas 1 aparição)")
print(f"{'='*80}")
print(f"Total de one-hits: {len(one_hits)}")
print(f"Probabilidade média (one-hits): {one_hits['prob_aparicao'].mean():.2%}")
print(f"Probabilidade média (múltiplas aparições): {df_predicoes[df_predicoes['aparicao_unica'] == 0]['prob_aparicao'].mean():.2%}")
print(f"\nOne-hits com maior probabilidade (top 10):")
print(one_hits.head(10)[['Artista', 'prob_aparicao_raw', 'prob_aparicao', 
                          'anos_desde_ultima', 'forca_musica']].to_string(index=False))

print(f"\n{'='*80}")
print(f"Top 20 músicas com MAIOR probabilidade de aparecer em {ano_predicao}:")
print(f"{'='*80}")
print(df_predicoes.head(20)[['Artista', 'Genero', 'prob_aparicao', 
                                   'num_aparicoes_historicas', 'streak_anos', 
                                   'apareceu_ano_anterior', 'aparicao_unica']].to_string(index=False))

# Criar visualização comparativa
print("\n{'='*80}")
print("Comparando probabilidades RAW vs AJUSTADAS")
print("{'='*80}")

# Comparar diferenças para one-hits
one_hits_sorted = one_hits.copy()
one_hits_sorted['diferenca'] = one_hits_sorted['prob_aparicao_raw'] - one_hits_sorted['prob_aparicao']
one_hits_sorted = one_hits_sorted.sort_values('diferenca', ascending=False)

print("\nOne-hits com MAIOR ajuste (penalidade):")
print(one_hits_sorted.head(10)[['Artista', 'prob_aparicao_raw', 'prob_aparicao', 
                                 'diferenca', 'anos_desde_ultima']].to_string(index=False))

## 8. Visualização

In [ ]:
# ============================================================================
# 8. VISUALIZAÇÕES
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 5))

# 1. Distribuição de probabilidades
axes[0].hist(df_predicoes['prob_aparicao'], bins=30, edgecolor='black')
axes[0].set_xlabel('Probabilidade de Aparição')
axes[0].set_ylabel('Número de Músicas')
axes[0].set_title('Distribuição de Probabilidades de Aparição')
axes[0].axvline(0.5, color='red', linestyle='--', label='Threshold')
axes[0].legend()

# 2. Relação entre número de aparições e probabilidade
df_prob_agg = df_prob.groupby('num_aparicoes')['apareceu_proximo_ano'].mean()
axes[1].plot(df_prob_agg.index, df_prob_agg.values, marker='o')
axes[1].set_xlabel('Número de Aparições Anteriores')
axes[1].set_ylabel('Taxa de Aparição no Próximo Ano')
axes[1].set_title('Histórico vs Probabilidade de Aparição')
axes[1].grid(True, alpha=0.3)

## 9. Exportação

In [ ]:
df_predicoes.to_csv("../data/prob_proximo_ano.csv", index=False)